<a href="https://colab.research.google.com/github/angelms2003/FernandezMartinezPolo-EML-RL/blob/main/Entornos_Complejos/sarsa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SARSA

*Description*: En este notebook se desarrolla la implementación del método de **SARSA (State-Action-Reward-State-Action)**, y se emplea sobre el entorno Taxi-v3 de Gymnasium.


    Autores: David Fernández Expósito
             Ángel Martínez Sánchez
             Javier Polo Gambín

    Emails: dfernandezexposito@um.es
            angel.martinezs@um.es
            javier.polog@um.es
            
    Date: 2026/02/25

Empezamos instalando e importando las librerías necesarias. También definimos los dispositivos donde se ejecutará el notebook y la semilla que vamos a usar para asegurar reproducibilidad.

In [ ]:
%%capture
!pip install gymnasium

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import gymnasium as gym
import torch
import gc
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SEED = 123

# NumPy
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Python
os.environ["PYTHONHASHSEED"] = str(SEED)

## Agente

Para implementar los distintos métodos de aprendizaje estudiados en la asignatura, hemos seguido las recomendaciones de Gymnasium para la creación de agentes y la generación de episodios, tal como se indicaba en la definición de la práctica. La idea central ha sido encapsular toda la lógica de interacción y aprendizaje en una clase `Agente`, adaptable a los diferentes algoritmos que se desean evaluar.

En este trabajo nos centramos en **SARSA**, un método de **Diferencias Temporales (TD)** de tipo **On-Policy**. A diferencia de los métodos Monte Carlo, que deben esperar al final de un episodio completo para actualizar los valores $Q(S,A)$, SARSA realiza actualizaciones **paso a paso** (*bootstrapping*), lo que le permite aprender de manera incremental a partir de cada transición individual.

El nombre del algoritmo proviene de la quíntupla de eventos que utiliza en cada actualización: $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$. La regla de actualización es:

$$
Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \, Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t) \right]
$$

donde $\alpha$ es la tasa de aprendizaje y $\gamma$ es el factor de descuento. Al ser un método On-Policy, SARSA evalúa y mejora la misma política que utiliza para tomar decisiones. Esto implica que la acción $A_{t+1}$ del siguiente estado se elige siguiendo la política actual (típicamente $\epsilon$-greedy), y es esa misma acción la que se usa para calcular el *TD target*.

En cuanto a la exploración, el agente utiliza una política **$\epsilon$-greedy** que combina explotación de la acción con mayor valor esperado con exploración aleatoria. Además, se implementa un mecanismo de **decaimiento de $\epsilon$** (*epsilon decay*), que reduce progresivamente la probabilidad de exploración a lo largo del entrenamiento. Esto permite una fase inicial de exploración intensa seguida de una consolidación progresiva hacia políticas más explotadoras.

La principal ventaja de SARSA frente a Monte Carlo radica en la eficiencia de sus actualizaciones: al corregir los valores $Q$ tras cada acción (y no al final del episodio), el agente puede penalizar trayectorias ineficientes de forma inmediata, lo que resulta especialmente beneficioso en entornos con episodios largos como Taxi-v3.

In [ ]:
class SARSAAgent:
    def __init__(self, env: gym.Env, alpha: float, gamma: float, epsilon: float, epsilon_decay: float, epsilon_min: float):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

        self.n_states = env.observation_space.n
        self.n_actions = env.action_space.n
        self.q_table = np.zeros((self.n_states, self.n_actions))

    def choose_action(self, state):
        """Política Epsilon-Greedy"""
        if np.random.uniform(0, 1) < self.epsilon:
            return self.env.action_space.sample()  # Exploración
        else:
            return np.argmax(self.q_table[state, :])  # Explotación

    def update(self, state, action, reward, next_state, next_action, done):
        """
        Actualización On-Policy (SARSA).
        A diferencia de Q-Learning, usamos el Q-valor de la acción que REALMENTE
        vamos a tomar (next_action), no el máximo.
        """
        td_target = reward + self.gamma * self.q_table[next_state, next_action] * (not done)
        td_error = td_target - self.q_table[state, action]
        self.q_table[state, action] += self.alpha * td_error

    def decay_exploration(self):
        """Decaimiento del epsilon al final del episodio"""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def get_q_values(self):
        return self.q_table

## Esquema de aprendizaje

Ahora implementamos el proceso de aprendizaje. La función `train_agent` ejecuta el bucle de entrenamiento de SARSA durante un número determinado de episodios.

A diferencia del esquema de Monte Carlo, donde la actualización se realizaba al final de cada episodio completo, aquí la actualización se realiza **en cada paso** de la interacción con el entorno. Esto constituye la diferencia fundamental entre ambas familias de métodos.

El flujo de SARSA es el siguiente: antes de entrar al bucle interno del episodio, se elige la primera acción $A_0$ según la política $\epsilon$-greedy. Dentro del bucle, se ejecuta la acción, se observa la transición $(S_t, A_t, R_{t+1}, S_{t+1})$, se elige la siguiente acción $A_{t+1}$ (también según la política actual), y se actualiza $Q(S_t, A_t)$ utilizando la quíntupla completa. Al finalizar cada episodio se aplica el decaimiento de $\epsilon$ y se registran las métricas de recompensa acumulada y longitud del episodio.

In [ ]:
def train_agent(agent, num_episodes=5000):
    stats_rewards = []
    stats_lengths = []

    for ep in tqdm(range(num_episodes)):
        state, info = agent.env.reset(seed=SEED)

        # SARSA: Elegimos la primera acción ANTES de entrar al bucle
        action = agent.choose_action(state)

        total_reward = 0
        steps = 0
        done = False

        while not done:
            # Ejecutamos la acción
            next_state, reward, terminated, truncated, info = agent.env.step(action)
            done = terminated or truncated

            # SARSA: Elegimos la siguiente acción (A') basándonos en la política actual
            next_action = agent.choose_action(next_state)

            # Actualizamos usando S, A, R, S', A'
            agent.update(state, action, reward, next_state, next_action, done)

            # Avanzamos el estado y la acción
            state = next_state
            action = next_action

            total_reward += reward
            steps += 1

        agent.decay_exploration()
        stats_rewards.append(total_reward)
        stats_lengths.append(steps)

    return stats_rewards, stats_lengths

## Funciones auxiliares

Ahora vamos a definir una serie de funciones auxiliares que nos van a servir para mostrar resultados y realizar el análisis.

La primera función que definimos la usaremos una vez entrenado el agente, de forma que podamos evaluar el aprendizaje llevado a cabo.

In [ ]:
def capture_optimal_behavior(agente, limit_steps=100):
    """
    Graba un episodio completo siguiendo la política óptima del agente
    y devuelve las métricas de rendimiento junto con los frames de video.
    """
    visual_frames = []
    current_state, _ = agente.env.reset(seed=SEED)

    accumulated_reward = 0.0
    steps_count = 0
    is_finished = False

    while not is_finished and steps_count < limit_steps:
        # 1. Capturar el estado visual actual
        img_frame = agente.env.render()
        visual_frames.append(img_frame)

        # 2. Decidir acción basada en la política objetivo (greedy)
        chosen_action = np.argmax(agente.get_q_values()[current_state, :])

        # 3. Ejecutar transición en el entorno
        next_s, reward, terminated, truncated, _ = agente.env.step(chosen_action)

        # 4. Actualizar contadores y estado
        accumulated_reward += reward
        current_state = next_s
        steps_count += 1
        is_finished = terminated or truncated

    # Capturar el último frame tras el fin del episodio
    visual_frames.append(agente.env.render())

    return accumulated_reward, steps_count, visual_frames

Las siguiente funciones mostrarán las gráficas de aprendizaje y longitud de los episodios una vez realizado el aprendizaje de los agentes.

La longitud del episodio es un medidor de rendimiento interesante porque no solo indica si el agente alcanza la meta, sino también **cómo de eficiente es la política aprendida**. En entornos donde existe una ruta óptima, la convergencia hacia un número estable y bajo de pasos suele reflejar que el agente ha aprendido un comportamiento estructurado y cercano al óptimo.

Además, esta métrica permite interpretar mejor los resultados: episodios muy cortos pueden indicar caídas tempranas en estados terminales negativos, mientras que episodios largos pueden reflejar exploración excesiva o movimientos erráticos. Por ello, la longitud del episodio complementa a la recompensa promedio y ayuda a entender no solo si el agente aprende, sino **cómo está aprendiendo**.

In [ ]:
def compute_running_mean(series, window):
    """Calcula un suavizado tipo media deslizante sobre una serie temporal."""
    kernel = np.full(window, 1.0 / window)
    return np.convolve(series, kernel, mode="valid")


def draw_multiple_learning_curves(results_dict, window=100):
    """
    Representa varias curvas de entrenamiento en el mismo gráfico.

    results_dict:
        Diccionario donde:
        clave -> nombre experimento/agente
        valor -> lista con historial de métricas
    """
    fig, ax = plt.subplots(figsize=(10, 4))
    palette = ["darkgreen", "darkred", "navy", "orange"]

    for idx, (experiment_name, history) in enumerate(results_dict.items()):
        color = palette[idx % len(palette)]

        # Señal original (transparente)
        ax.plot(history, alpha=0.1, color=color)

        # Tendencia suavizada (media móvil)
        smoothed = compute_running_mean(history, window)
        ax.plot(
            np.arange(len(smoothed)),
            smoothed,
            linewidth=2,
            color=color,
            label=experiment_name
        )

    ax.set_title("Comparativa de rendimiento (SARSA)")
    ax.set_xlabel("Número de episodio")
    ax.set_ylabel("Recompensa por episodio")
    ax.legend()
    ax.grid(True, alpha=0.4)
    plt.show()


def draw_episode_length_comparison(length_dict, window=100):
    """
    Compara la evolución de longitud de episodio entre varios experimentos.
    """
    fig, ax = plt.subplots(figsize=(10, 4))
    palette = ["darkgreen", "darkred", "navy", "orange"]
    
    for idx, (label, values) in enumerate(length_dict.items()):
        color = palette[idx % len(palette)]

        # Señal original
        ax.plot(values, alpha=0.1, color=color)

        # Tendencia suavizada
        smoothed = compute_running_mean(values, window)
        ax.plot(
            np.arange(len(smoothed)),
            smoothed,
            linewidth=2,
            color=color,
            label=label
        )

    ax.set_title("Comparativa de longitudes de episodio (SARSA)")
    ax.set_xlabel("Índice de episodio")
    ax.set_ylabel("Número de pasos")
    ax.legend()
    ax.grid(True, alpha=0.4)
    plt.show()

Las siguientes funciones servirán para visualizar los resultados con imágenes y gifs del comportamiento del agente.

In [ ]:
import seaborn as sns
import imageio
import base64
from IPython.display import HTML
import matplotlib.pyplot as plt

def get_taxi_qtable_directions(qtable, env):
    """
    Extrae la matriz de valores Q máximos y los símbolos de las mejores acciones
    para una configuración específica del pasajero y destino en Taxi-v3.
    """
    state, _ = env.reset(seed=SEED)
    _, _, pass_idx, dest_idx = env.unwrapped.decode(state)
    q_max_grid = np.zeros((5, 5))
    directions_grid = np.empty((5, 5), dtype=object)

    # Mapeo de acciones para Taxi-v3
    # 0: Sur, 1: Norte, 2: Este, 3: Oeste, 4: Recoger (P), 5: Dejar (D)
    action_symbols = {0: '↓', 1: '↑', 2: '→', 3: '←', 4: 'P', 5: 'D'}

    for row in range(5):
        for col in range(5):
            state = env.unwrapped.encode(row, col, pass_idx, dest_idx)
            best_action = int(np.argmax(qtable[state]))
            max_q_value = np.max(qtable[state])

            q_max_grid[row, col] = max_q_value
            directions_grid[row, col] = action_symbols[best_action]

    return q_max_grid, directions_grid


def plot_taxi_q_values_map(qtable, env):
    '''
    Plotea un mapa de calor (Heatmap) de la política aprendida y los valores Q máximos.
    '''
    q_max_grid, directions_grid = get_taxi_qtable_directions(qtable, env)

    plt.figure(figsize=(7, 6))
    ax = sns.heatmap(
        q_max_grid,
        annot=directions_grid,
        fmt="",
        cmap=sns.color_palette("Blues", as_cmap=True),
        linewidths=1.5,
        linecolor="black",
        xticklabels=[],
        yticklabels=[],
        cbar_kws={'label': 'Max Q-Value estimado'},
        annot_kws={"fontsize": 18, "weight": "bold", "color": "black"},
    )
    ax.set_title("Learned Q-values\nArrows and letters (P/D) represent best action", fontsize=14)

    for _, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
        spine.set_color("black")

    plt.tight_layout()
    plt.show()


def create_gif_from_frames(frame_list, output_path="agent_taxi.gif"):
    """Genera un GIF animado a partir de una lista de imágenes."""
    with imageio.get_writer(output_path, mode="I") as gif_writer:
        for frame in frame_list:
            gif_writer.append_data(frame)
    return output_path


def show_gif_in_notebook(gif_file_path):
    """Inserta un GIF en una celda de Jupyter Notebook o Colab."""
    with open(gif_file_path, "rb") as f:
        gif_bytes = f.read()
    b64_str = base64.b64encode(gif_bytes).decode("utf-8")
    return HTML(f'<img src="data:image/gif;base64,{b64_str}" />')

## Entorno Taxi-v3

A continuación, creamos el entorno "Taxi-v3" de Gymnasium con el que trabajaremos.

`Taxi-v3` es un entorno estructurado de 500 estados utilizado para evaluar la escalabilidad de algoritmos tabulares.  

**Características del Entorno:**
* **Estados (500):** Combina 25 posiciones del taxi (cuadrícula 5x5), 5 posiciones posibles del pasajero (4 ubicaciones + dentro del taxi) y 4 destinos posibles.
* **Acciones (6):** Moverse al Sur, Norte, Este, Oeste, Recoger pasajero (Pickup) y Dejar pasajero (Dropoff).
* **Recompensas:**
  * **-1** por cada paso ejecutado (presiona al agente a encontrar la ruta más rápida).
  * **+20** por dejar al pasajero en su destino correctamente.
  * **-10** por ejecutar erróneamente *Pickup* o *Dropoff* en ubicaciones no válidas.

In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")

Ahora creamos los diferentes agentes que usaremos. El objetivo principal de esta serie de experimentos consiste en comprobar empíricamente si la actualización paso a paso (*bootstrapping*) de SARSA logra superar los resultados obtenidos previamente por los enfoques de Monte Carlo. Además, se busca caracterizar el comportamiento del algoritmo y su sensibilidad respecto a dos de sus hiperparámetros fundamentales: la tasa de aprendizaje ($\alpha$) y el factor de descuento ($\gamma$).

Para garantizar una comparativa metodológicamente justa y rigurosa frente a los ensayos de Monte Carlo, se ha implementado una estrategia de exploración estructurada a través de un mecanismo de decaimiento del parámetro $\epsilon$ ($\epsilon$-decay). Dicho decaimiento se inicializa en $\epsilon_0 = 1.0$, con una reducción en un factor de $0.990$ por episodio hasta alcanzar una cota inferior de $\epsilon_{min} = 0.01$. El entrenamiento se extiende durante $5000$ episodios continuos.

Se han diseñado y evaluado un total de cuatro configuraciones diferentes a partir del producto cruzado de los hiperparámetros de interés:

- $\alpha \in \{0.1, 0.5\}$: se exploran una tasa de aprendizaje conservadora y una agresiva.
- $\gamma \in \{0.90, 0.99\}$: se compara un horizonte de recompensas moderado con uno amplio.

Este diseño permite estudiar:

- La influencia de la tasa de aprendizaje en la velocidad de convergencia y la estabilidad.
- El impacto del horizonte de recompensas en la calidad de la política aprendida.
- La interacción entre ambos hiperparámetros en un entorno determinista con recompensas dispersas como Taxi-v3.

In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")
n_episodes = 5000

# Parámetros de exploración comunes a todos los agentes
epsilon_0 = 1.0
decay = 0.990
epsilon_min = 0.01

# Estudio conjunto de SARSA: α ∈ {0.1, 0.5} × γ ∈ {0.90, 0.99}
agents = {
    "α=0.1, γ=0.99": SARSAAgent(env, alpha=0.1, gamma=0.99, epsilon=epsilon_0, epsilon_decay=decay, epsilon_min=epsilon_min),
    "α=0.5, γ=0.99": SARSAAgent(env, alpha=0.5, gamma=0.99, epsilon=epsilon_0, epsilon_decay=decay, epsilon_min=epsilon_min),
    "α=0.1, γ=0.90": SARSAAgent(env, alpha=0.1, gamma=0.90, epsilon=epsilon_0, epsilon_decay=decay, epsilon_min=epsilon_min),
    "α=0.5, γ=0.90": SARSAAgent(env, alpha=0.5, gamma=0.90, epsilon=epsilon_0, epsilon_decay=decay, epsilon_min=epsilon_min),
}

results_rewards = {}
results_lengths = {}

for name, agent in agents.items():
    print(f"Entrenando: {name}...")
    rewards, lengths = train_agent(agent, num_episodes=n_episodes)
    results_rewards[name] = rewards
    results_lengths[name] = lengths

## Resultados

Guardamos los resultados obtenidos en diccionarios para pasárselos a las funciones auxiliares definidas anteriormente para plotear los resultados.

In [ ]:
np.savez('sarsa_results.npz', results_rewards=results_rewards, results_lengths=results_lengths)
print("Resultados guardados en 'sarsa_results.npz'")

In [ ]:
draw_multiple_learning_curves(results_rewards, window=100)

In [ ]:
draw_episode_length_comparison(results_lengths, window=100)

Las gráficas de recompensa por episodio y longitud de los episodios obtenidas para el entorno Taxi-v3 mediante SARSA reflejan una **mejora drástica** respecto a los resultados obtenidos previamente con los métodos Monte Carlo, tanto en el rendimiento final sostenido como en la velocidad de convergencia.

Todas las combinaciones de hiperparámetros puestas a prueba convergen de forma consistente hacia la política óptima en las primeras 500 iteraciones del entrenamiento. Esta rápida convergencia resalta la principal ventaja de la aproximación de Diferencias Temporales frente a Monte Carlo: al aplicar las actualizaciones tras cada acción de movimiento en lugar de postergarlas al final de la trayectoria, SARSA permite al agente penalizar trayectos o bucles ineficientes de manera inmediata, evitando el agotamiento de los límites temporales del episodio.

Analizando la parametrización, se comprueba que el aprendizaje más robusto se obtiene con la configuración $\alpha = 0.5$ y $\gamma = 0.99$. Esta observación posee una interpretación muy intuitiva basada en la estructura del entorno. Por un lado, una tasa de aprendizaje alta ($\alpha = 0.5$) resulta sumamente beneficiosa debido a la naturaleza determinista de las transiciones en Taxi-v3: al carecer de ruido estocástico (no hay probabilidades de desplazamientos erráticos), un ajuste más rápido de los valores Q genera un aprendizaje confiable sin riesgo de oscilaciones inestables.

Por otro lado, la adopción de un factor de descuento alto ($\gamma = 0.99$) es crítica y coherente con la estructura de recompensas del entorno. La única recompensa estrictamente positiva (+20) se obtiene al completar la tarea completa (depositar al pasajero en destino); todos los pasos previos únicamente generan penalizaciones de -1. Un $\gamma$ próximo a la unidad permite una propagación no degradada de este estímulo de bonificación hacia todos los estados y acciones precursoras, lo que resulta en rutas más óptimas y una estabilización superior.

En cuanto a la longitud de los episodios, se observa un patrón coherente: al inicio del entrenamiento los episodios alcanzan frecuentemente el límite máximo de pasos, pero a medida que el aprendizaje progresa, el número de pasos necesarios para completar cada episodio disminuye drásticamente, confirmando que el agente encuentra rutas cada vez más eficientes para recoger y transportar al pasajero.

Ahora vamos a visualizar la política aprendida, usando las funciones auxiliares que muestran visualmente la política y generan gifs con el comportamiento del agente.

El entorno Taxi-v3 define el estado como una tupla de cuatro componentes:

$$
(row, col, passenger\_idx, destination\_idx)
$$

Esto implica que el espacio de estados es de dimensión 4, por lo que no es posible representar directamente la política completa en un mapa bidimensional.

Para poder visualizar el comportamiento aprendido por el agente, se fija la posición del pasajero y el destino, representando únicamente la política asociada a esa sección concreta del espacio de estados. De esta forma, se construye un mapa 5×5 que muestra la acción greedy seleccionada por el agente para cada posición del taxi bajo esas condiciones fijas.

Es importante destacar que esta representación no muestra la política global del agente, sino una proyección bidimensional condicionada a un estado específico del pasajero y del destino.

In [ ]:
# Mostramos el mapa de la política aprendida para el mejor agente (α=0.5, γ=0.99)
best_agent = agents["α=0.5, γ=0.99"]
plot_taxi_q_values_map(best_agent.get_q_values(), env)

Comparamos también la política del agente con peor configuración para observar las diferencias.

In [ ]:
# Política del agente con α=0.1, γ=0.90
plot_taxi_q_values_map(agents["α=0.1, γ=0.90"].get_q_values(), env)

En los mapas de calor se aprecia que el agente con la mejor configuración ($\alpha = 0.5$, $\gamma = 0.99$) muestra valores Q más altos y coherentes, con flechas de dirección que reflejan claramente el intento de desplazarse hacia la posición del pasajero. En cambio, el agente con peor configuración presenta valores Q más uniformes y un patrón de acciones menos definido, indicando un aprendizaje de menor calidad.

## Visualización del agente en acción

Al igual que analizamos las métricas numéricas, es fundamental observar cualitativamente cómo se comporta el agente tras el entrenamiento. A continuación, ejecutamos un episodio utilizando la tabla Q aprendida (explotación pura, sin exploración) y capturamos los fotogramas para generar un GIF.

In [ ]:
# Ejecutar un episodio utilizando la política greedy del mejor agente
best_agent = agents["α=0.5, γ=0.99"]
reward, len_episode, frames = capture_optimal_behavior(best_agent)

print(f"Episodio de prueba finalizado.")
print(f"Recompensa total: {reward}")
print(f"Pasos realizados: {len_episode}")

# Crear y mostrar el GIF
path_gif = "SARSA_Taxi_Optimal.gif"
create_gif_from_frames(frames, path_gif)
print("Archivo guardado en:", path_gif)
show_gif_in_notebook(path_gif)

### ¿Por qué la gráfica $f(t) = len(episodio_t)$ es un buen indicador de aprendizaje?

Hemos incluido la evolución temporal de la longitud del episodio (pasos por episodio). Esta métrica es un indicador de aprendizaje excelente, a veces incluso superior a la recompensa bruta, por dos motivos:

1. **Eficiencia de la ruta:** En entornos como *Taxi-v3*, donde cada movimiento penaliza con -1, la única forma de maximizar la recompensa es minimizando los pasos. La curva de longitud nos muestra visualmente cómo el agente deja de deambular de forma aleatoria (exploración) y comienza a trazar la ruta más corta hacia su objetivo.

2. **Aislamiento del ruido:** La recompensa total puede dar saltos bruscos (por ejemplo, el +20 al dejar al pasajero o el -10 por un error). Sin embargo, la métrica de pasos decrece de forma mucho más suave y monótona, permitiendo observar con mayor precisión en qué momento exacto convergen las políticas.

## Posibles mejoras y líneas de investigación futuras

**Comparación con Q-Learning:** Dado que SARSA es un método On-Policy y Q-Learning es Off-Policy, sería interesante comparar directamente ambos algoritmos bajo las mismas condiciones experimentales para cuantificar las diferencias en velocidad de convergencia y estabilidad. Esta comparación se realiza en el notebook correspondiente a Q-Learning.

**Análisis de la interacción con entornos estocásticos:** En un entorno determinista como Taxi-v3, las diferencias entre SARSA y Q-Learning se difuminan conforme $\epsilon$ decae. Sería relevante evaluar ambos algoritmos en entornos con transiciones estocásticas (como *CliffWalking*) donde la naturaleza On-Policy de SARSA podría ofrecer ventajas significativas en términos de seguridad.

**Esquemas alternativos de decaimiento de $\epsilon$:** Se ha utilizado un esquema multiplicativo constante para el decaimiento. Podrían explorarse esquemas adaptativos o basados en el rendimiento del agente para optimizar el equilibrio exploración-explotación.

**Aumento del número de episodios y análisis estadístico:** Repetir cada configuración con distintas semillas aleatorias permitiría un análisis estadístico más riguroso, con media y desviación estándar del rendimiento, reduciendo el impacto de la estocasticidad.